
# Single-turn simulation (LLM seeding, full grid)

Runs the full emotion/strategy grid with LLM-generated seeds (5 trials each), includes baseline, and computes confidence intervals for follow-up emotions.


In [ ]:

from pathlib import Path
import json
import pandas as pd
from dynamic_conversation import (
    SingleTurnSimulator,
    SimulationConfig,
    ResponseStrategy,
    compute_transition_confidence_intervals,
)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

cfg = SimulationConfig(
    model="gpt-4o-mini",
    max_tokens=800,
    temperature=1.0,
    brevity_hint="Reply in 1-2 sentences, keep emotion visible."
)

emotions = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
strategies = list(ResponseStrategy)
style_mod = "concise and emotionally attuned"

sim = SingleTurnSimulator(
    use_gpu=False,
    config=cfg,
    prompt_for_key=True,
)

out_prefix = Path("results/full_llm_single_turn")
out_prefix.parent.mkdir(exist_ok=True)

print("Running LLM-seeded full grid...")
df_llm = sim.run_batch(
    emotions=emotions,
    strategies=strategies,
    runs_per_pair=5,
    style_modifier=style_mod,
    use_llm_seed=True,
    include_baseline=True,
    save_csv=out_prefix.with_suffix(".csv"),
    save_heatmap=out_prefix.with_suffix("_heatmap.png"),
)

meta_path = out_prefix.with_suffix(".meta.json")
meta = json.load(open(meta_path)) if meta_path.exists() else {}
print("Meta:", meta)


In [ ]:

# Compute Wilson CIs for follow-up emotion proportions per (intended_emotion, strategy)
ci_df = compute_transition_confidence_intervals(df_llm, confidence=0.95)
ci_path = out_prefix.with_suffix("_ci.csv")
ci_df.to_csv(ci_path, index=False)
ci_df.head()


## Heatmap

In [ ]:

from IPython.display import Image
hm = out_prefix.with_suffix("_heatmap.png")
if hm.exists():
    display(Image(filename=str(hm)))
else:
    print("Heatmap not found:", hm)
